# 06 — Narrative Filers: Re-clustering a Cleaner Population

Notebook 05 found that the dominant axis in the complaint corpus isn't product category — it's complaint strategy. A large chunk of the dataset is people filing using legal templates (FCRA, FDCPA statutory citations, credit repair playbooks). Those complaints cluster by template, not by financial situation.

That's interesting, but it's not what we actually want to analyse. The goal is to find patterns in real consumer experiences — what's actually happening to people, not which legal playbook they followed.

So this notebook does two things:

1. Re-runs the full pipeline on a larger sample (5k complaints instead of 2k)
2. Filters to narrative filers only, then re-clusters

The communities that emerge from the filtered population should reflect actual financial situations — the kind of patterns an investigator would care about.


## Setup


In [9]:
import sys
from pathlib import Path

sys.path.append(str(Path("..").resolve()))

import numpy as np
import pandas as pd
import networkx as nx
import community as community_louvain
import plotly.express as px
import plotly.graph_objects as go
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer

from src.embeddings import load_model, generate_embeddings
from src.retrieval import get_neighbors


## Load data

Scaling up to 15k complaints. Same parquet, same column, just taking more rows.

The larger sample matters for the narrative filer population specifically — after filtering out templates we expect roughly 30-40% of complaints to survive, so 5k gives us ~1,500-2,000 narrative complaints to re-cluster. That's enough to find meaningful structure.


In [10]:
SAMPLE_SIZE = 15000

df = pd.read_parquet("../data/processed/complaints_50k.parquet")

df_sample = df.iloc[:SAMPLE_SIZE].copy()

texts = (
    df_sample["Consumer complaint narrative"]
    .fillna("")
    .astype(str)
    .tolist()
)

# We'll need these metadata columns later for community characterisation
META_COLS = ["Product", "Issue", "Sub-issue", "Company", "Date received", "Complaint ID"]

print(f"Loaded {len(df_sample):,} complaints")
print(f"Complaints with narrative: {df_sample['Consumer complaint narrative'].notna().sum():,}")
print(f"\nAvailable columns: {list(df_sample.columns)}")


Loaded 15,000 complaints
Complaints with narrative: 6,150

Available columns: ['Date received', 'Product', 'Sub-product', 'Issue', 'Sub-issue', 'Consumer complaint narrative', 'Company public response', 'Company', 'State', 'ZIP code', 'Tags', 'Consumer consent provided?', 'Submitted via', 'Date sent to company', 'Company response to consumer', 'Timely response?', 'Consumer disputed?', 'Complaint ID']


In [11]:
df_sample = df.iloc[:SAMPLE_SIZE].copy()
df_sample = df_sample[df_sample["Consumer complaint narrative"].notna()].copy()
df_sample = df_sample[df_sample["Consumer complaint narrative"].str.strip() != ""].copy()
df_sample = df_sample.reset_index(drop=True)

texts = df_sample["Consumer complaint narrative"].astype(str).tolist()

print(f"Complaints with narrative: {len(df_sample):,}")

Complaints with narrative: 6,150


## Generate embeddings

This will take a couple of minutes at 5k. Same model as before.


In [12]:
model = load_model()

embeddings = generate_embeddings(texts, model)

print(f"Embeddings shape: {embeddings.shape}")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/193 [00:00<?, ?it/s]

Embeddings shape: (6150, 384)


## Run the full pipeline on the 15k sample

Before filtering, let's run Louvain on the full 15k to get community assignments. We need these to apply the template/narrative labels from notebook 05.

Same parameters: k=10, no threshold, resolution=1.0.


In [13]:
def build_knn_graph(embeddings, k):
    G = nx.Graph()
    G.add_nodes_from(range(len(embeddings)))

    for i in range(len(embeddings)):
        neighbors, similarities = get_neighbors(
            embeddings,
            query_index=i,
            k=k + 1
        )
        for neighbor in neighbors:
            if neighbor == i:
                continue
            sim = similarities[neighbor]
            if G.has_edge(i, neighbor):
                existing = G[i][neighbor]["weight"]
                G[i][neighbor]["weight"] = (existing + sim) / 2
            else:
                G.add_edge(i, neighbor, weight=sim)

    return G


K = 10
G_full = build_knn_graph(embeddings, K)

print(f"Graph: {G_full.number_of_nodes():,} nodes, {G_full.number_of_edges():,} edges")


Graph: 6,150 nodes, 49,559 edges


In [14]:
SEED = 42

partition_full = community_louvain.best_partition(
    G_full,
    weight="weight",
    resolution=1.0,
    random_state=SEED
)

community_ids_full = np.array([partition_full[i] for i in range(len(partition_full))])
n_communities_full = len(set(community_ids_full))
modularity_full = community_louvain.modularity(partition_full, G_full, weight="weight")

sizes_full = sorted(Counter(community_ids_full).values(), reverse=True)

print(f"Communities: {n_communities_full}")
print(f"Modularity:  {modularity_full:.4f}")
print(f"Sizes: {sizes_full}")


Communities: 24
Modularity:  0.6999
Sizes: [1005, 865, 786, 719, 424, 365, 343, 285, 238, 203, 154, 118, 100, 73, 72, 70, 65, 57, 56, 42, 37, 29, 29, 15]


In [15]:
resolutions = [0.5, 0.75, 1.0, 1.25, 1.5, 1.75, 2.0, 2.5, 3.0]
res_results = []

for res in resolutions:
    p = community_louvain.best_partition(
        G_full, weight="weight", resolution=res, random_state=SEED
    )
    mod = community_louvain.modularity(p, G_full, weight="weight")
    comm_sizes = sorted(Counter(p.values()).values(), reverse=True)
    res_results.append({
        "resolution": res,
        "n_communities": len(comm_sizes),
        "modularity": round(mod, 4),
        "largest": comm_sizes[0],
        "largest_pct": round(100 * comm_sizes[0] / len(embeddings), 1),
        "median_size": int(np.median(comm_sizes)),
        "singletons": sum(1 for s in comm_sizes if s == 1),
    })

res_df = pd.DataFrame(res_results)
print(res_df.to_string(index=False))

 resolution  n_communities  modularity  largest  largest_pct  median_size  singletons
       0.50             26      0.6887     1078         17.5           99           0
       0.75             28      0.6921     1121         18.2          104           0
       1.00             24      0.6999     1005         16.3          109           0
       1.25             29      0.6988      989         16.1          129           0
       1.50             31      0.6896      740         12.0          127           0
       1.75             36      0.6820      737         12.0          102           0
       2.00             40      0.6804      724         11.8           97           0
       2.50             46      0.6660      456          7.4           94           0
       3.00             57      0.6592      418          6.8           86           0


## Label communities: template vs narrative

In notebook 05 we ran this on 2k complaints and found 15 communities. Reading the central complaints and TF-IDF keywords, the split was clear:

**Template filers** — legal citation language, FCRA/FDCPA statutes, credit repair playbooks. The complaint is a legal instrument, not a personal account.

**Narrative filers** — people describing something that happened to them in plain language. The complaint is a story.

Now we're running on 5k so the community structure may differ — different number of communities, different boundaries. We need to re-inspect and re-assign labels.

The cell below runs the same inspection tools from notebook 05: TF-IDF keywords and the most central complaint per community. Read them and update `NARRATIVE_COMMUNITIES` accordingly.


In [16]:
# TF-IDF keywords per community — read these to assign labels
community_docs = {
    comm_id: " ".join(
        texts[i] for i in range(len(texts))
        if community_ids_full[i] == comm_id and texts[i]
    )
    for comm_id in sorted(set(community_ids_full))
}

comm_ids_ordered = sorted(community_docs.keys())
corpus = [community_docs[c] for c in comm_ids_ordered]

vec = TfidfVectorizer(
    max_features=5000,
    stop_words="english",
    ngram_range=(1, 2),
    sublinear_tf=True
)
tfidf = vec.fit_transform(corpus)
feature_names = vec.get_feature_names_out()

print("TF-IDF keywords per community — use these to assign template/narrative labels\n")
for i, comm_id in enumerate(comm_ids_ordered):
    row = tfidf[i].toarray().flatten()
    top_idx = row.argsort()[::-1][:10]
    keywords = ", ".join(feature_names[j] for j in top_idx)
    size = (community_ids_full == comm_id).sum()
    print(f"C{comm_id} (n={size:>4}): {keywords}")


TF-IDF keywords per community — use these to assign template/narrative labels

C0 (n= 786): physically verifiable, plaintiffs, alleged claims, metro format, verifiable document, standing accordance, statement claim, debt claim, recovery associates, medical debt
C1 (n= 865): 605b credit, reported identity, oh xxxx, xxxx oh, 00 00, send original, posted report, individual individual, authorized user, installment installment
C2 (n= 100): delete accounts, unjust, investigate accounts, report inaccurate, credit report, concerned, duty, credit, accounts inquiries, xxxx
C3 (n=1005): citibank, bank west, barclay, chase, bonus, balance transfer, kohl, bofa, chime, card number
C4 (n= 365): failure investigate, gives reason, reason immediately, days deleted, deleted promptly, promptly demand, caused information, unknown things, days gives, litigation stress
C5 (n= 719): freedom mortgage, phh, escrow, select portfolio, forgiveness, pmi, shellpoint, sps, portfolio servicing, loss mitigation
C6 (n= 

In [17]:
# Most central complaint per community
print("Most central complaint per community\n" + "="*65)

for comm_id in sorted(set(community_ids_full)):
    nodes_in_comm = [n for n, c in partition_full.items() if c == comm_id]
    subG = G_full.subgraph(nodes_in_comm)
    central_node = max(subG.degree(), key=lambda x: x[1])[0]
    size = len(nodes_in_comm)
    text = texts[central_node]
    print(f"\nC{comm_id} ({size} complaints) — node {central_node}")
    print(text[:300] if text else "[no narrative]")
    print("-"*65)


Most central complaint per community

C0 (786 complaints) — node 775
THIS COMPANY CAN NOT AND HAS NOT produced me my original bill from the original creditor they purchased my debt from and because of that they must to cease all collection efforts and remove this account from my credit report. REMEMBER I AM REQUESTING VALIDATION, NOT VERIFICATION!! This COMPLAINT is 
-----------------------------------------------------------------

C1 (865 complaints) — node 2825
Im submitting a complaint to you today to inform you I was the victim of identity theft. I researched on how to remove the fraudulent account in my report and found that I need to visit FEDERAL TRADE COMMISION or https : //www.ftc.gov to file a report and Per FCRA section 605b Credit Reporting Agenc
-----------------------------------------------------------------

C2 (100 complaints) — node 4661
The items that are reflected on my credit report are inaccurate. It is your duty to inform consumers about the things they need to 

## Assign labels

Based on the keywords and central complaints above, update the list below.

From notebook 05 the template communities were characterised by: USC/FCRA/FDCPA citations, statutory language ("pursuant to", "in accordance with"), legal argument structure, credit repair boilerplate.

Narrative communities were characterised by: institution names (Chase, Shellpoint, GM Financial), specific financial situations (escrow, forbearance, ATM fraud), plain language describing events.


In [18]:
# ── We update the narrative communities after reading the output above ──────────────────────────────
# List the community IDs that are narrative filers.
# Everything else is treated as template.

NARRATIVE_COMMUNITIES = [2, 3, 5, 12, 13, 16, 17, 18, 21]

# ─────────────────────────────────────────────────────────────────────────────

df_sample["community_full"] = community_ids_full
df_sample["filer_type"] = df_sample["community_full"].apply(
    lambda c: "narrative" if c in NARRATIVE_COMMUNITIES else "template"
)

counts = df_sample["filer_type"].value_counts()
print(counts.to_string())
print(f"\nNarrative filers: {counts.get('narrative', 0):,} ({counts.get('narrative', 0)/len(df_sample)*100:.1f}%)")
print(f"Template filers:  {counts.get('template', 0):,} ({counts.get('template', 0)/len(df_sample)*100:.1f}%)")


filer_type
template     3635
narrative    2515

Narrative filers: 2,515 (40.9%)
Template filers:  3,635 (59.1%)


## Filter to narrative filers

Dropping template filers and complaints without a narrative. What's left is the population we actually want to analyse.


In [19]:
df_narrative = (
    df_sample[
        (df_sample["filer_type"] == "narrative") &
        (df_sample["Consumer complaint narrative"].notna()) &
        (df_sample["Consumer complaint narrative"].str.strip() != "")
    ]
    .copy()
    .reset_index(drop=True)
)

narrative_texts = df_narrative["Consumer complaint narrative"].astype(str).tolist()
narrative_indices = df_narrative.index.tolist()

print(f"Narrative filer corpus: {len(df_narrative):,} complaints")
print(f"\nProduct distribution:")
print(df_narrative["Product"].value_counts().head(10).to_string())


Narrative filer corpus: 2,515 complaints

Product distribution:
Product
Credit reporting, credit repair services, or other personal consumer reports    877
Credit card or prepaid card                                                     409
Mortgage                                                                        374
Checking or savings account                                                     322
Money transfer, virtual currency, or money service                              194
Debt collection                                                                 111
Student loan                                                                    101
Vehicle loan or lease                                                            74
Payday loan, title loan, or personal loan                                        53


## Re-embed the narrative corpus

We need fresh embeddings for just this population. The filtered corpus will have a different geometric structure — without the legal citation clusters dominating the space, the remaining semantic variation should be more visible.


In [20]:
narrative_embeddings = generate_embeddings(narrative_texts, model)

print(f"Narrative embeddings: {narrative_embeddings.shape}")


Batches:   0%|          | 0/79 [00:00<?, ?it/s]

Narrative embeddings: (2515, 384)


## UMAP projection


In [21]:
import umap.umap_ as umap

reducer = umap.UMAP(
    n_neighbors=15,
    min_dist=0.1,
    metric="cosine",
    random_state=42
)

embedding_2d = reducer.fit_transform(narrative_embeddings)

print(f"2D projection: {embedding_2d.shape}")


c:\Users\pablo\anaconda3\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


2D projection: (2515, 2)


## Re-cluster the narrative population

Same approach as notebook 05. The communities here should reflect actual financial situations, not legal strategies.


In [22]:
G_narrative = build_knn_graph(narrative_embeddings, K)

partition_narrative = community_louvain.best_partition(
    G_narrative,
    weight="weight",
    resolution=1.0,
    random_state=SEED
)

community_ids_narrative = np.array([partition_narrative[i] for i in range(len(partition_narrative))])
n_communities_narrative = len(set(community_ids_narrative))
modularity_narrative = community_louvain.modularity(partition_narrative, G_narrative, weight="weight")

sizes_narrative = sorted(Counter(community_ids_narrative).values(), reverse=True)

print(f"Communities: {n_communities_narrative}")
print(f"Modularity:  {modularity_narrative:.4f}")
print(f"Sizes: {sizes_narrative}")

df_narrative["community"] = community_ids_narrative


Communities: 17
Modularity:  0.6934
Sizes: [441, 403, 339, 316, 199, 111, 101, 98, 90, 75, 74, 69, 58, 56, 38, 27, 20]


## Visualise


In [23]:
plot_df = pd.DataFrame({
    "x": embedding_2d[:, 0],
    "y": embedding_2d[:, 1],
    "community": [f"C{c}" for c in community_ids_narrative],
    "product": df_narrative["Product"].tolist(),
    "issue": df_narrative["Issue"].tolist(),
    "company": df_narrative["Company"].tolist(),
    "text_preview": [t[:120] for t in narrative_texts]
})

fig = px.scatter(
    plot_df,
    x="x", y="y",
    color="community",
    hover_data={
        "product": True,
        "issue": True,
        "company": True,
        "text_preview": True,
        "x": False,
        "y": False
    },
    opacity=0.65,
    title=f"Narrative filer communities (n={len(df_narrative):,}, {n_communities_narrative} communities)"
)

fig.update_traces(marker=dict(size=5))
fig.update_layout(width=950, height=750)
fig.show()


## Characterise communities

Now we have `Issue` and `Company` as additional signals — much more informative than `Product` alone.


In [24]:
# Issue distribution per community
print("Top issues per community\n")
for comm_id in sorted(set(community_ids_narrative)):
    mask = community_ids_narrative == comm_id
    size = mask.sum()
    issues = df_narrative[mask]["Issue"].value_counts().head(3)
    print(f"C{comm_id} (n={size}):")
    for issue, count in issues.items():
        pct = 100 * count / size
        print(f"  {str(issue)[:60]:<60} {count:>4}  ({pct:.0f}%)")
    print()


Top issues per community

C0 (n=74):
  Incorrect information on your report                           40  (54%)
  Problem with a credit reporting company's investigation into   20  (27%)
  Attempts to collect debt not owed                               5  (7%)

C1 (n=403):
  Problem with a purchase shown on your statement                68  (17%)
  Fees or interest                                               29  (7%)
  Incorrect information on your report                           27  (7%)

C2 (n=58):
  Incorrect information on your report                            9  (16%)
  Problem with a purchase shown on your statement                 8  (14%)
  Problem when making payments                                    6  (10%)

C3 (n=90):
  Managing an account                                            20  (22%)
  Problem with a purchase shown on your statement                12  (13%)
  Fraud or scam                                                   8  (9%)

C4 (n=75):
  Managing an acco

In [25]:
# Company distribution per community
print("Top companies per community\n")
for comm_id in sorted(set(community_ids_narrative)):
    mask = community_ids_narrative == comm_id
    size = mask.sum()
    companies = df_narrative[mask]["Company"].value_counts().head(3)
    print(f"C{comm_id} (n={size}):")
    for company, count in companies.items():
        pct = 100 * count / size
        print(f"  {str(company)[:55]:<55} {count:>4}  ({pct:.0f}%)")
    print()


Top companies per community

C0 (n=74):
  TRANSUNION INTERMEDIATE HOLDINGS, INC.                    20  (27%)
  Experian Information Solutions Inc.                       19  (26%)
  EQUIFAX, INC.                                             16  (22%)

C1 (n=403):
  SYNCHRONY FINANCIAL                                       37  (9%)
  Bread Financial Holdings, Inc.                            35  (9%)
  AMERICAN EXPRESS COMPANY                                  34  (8%)

C2 (n=58):
  CAPITAL ONE FINANCIAL CORPORATION                         58  (100%)

C3 (n=90):
  JPMORGAN CHASE & CO.                                      82  (91%)
  CITIBANK, N.A.                                             2  (2%)
  SYNCHRONY FINANCIAL                                        2  (2%)

C4 (n=75):
  CITIBANK, N.A.                                            64  (85%)
  WELLS FARGO & COMPANY                                      2  (3%)
  TD BANK US HOLDING COMPANY                                 2  (3%)

C5 (n=

In [26]:
# TF-IDF keywords
narrative_community_docs = {
    comm_id: " ".join(
        narrative_texts[i] for i in range(len(narrative_texts))
        if community_ids_narrative[i] == comm_id
    )
    for comm_id in sorted(set(community_ids_narrative))
}

narrative_corpus = [narrative_community_docs[c] for c in sorted(narrative_community_docs.keys())]

vec2 = TfidfVectorizer(
    max_features=5000,
    stop_words="english",
    ngram_range=(1, 2),
    sublinear_tf=True
)
tfidf2 = vec2.fit_transform(narrative_corpus)
feature_names2 = vec2.get_feature_names_out()

print("TF-IDF keywords per community\n")
for i, comm_id in enumerate(sorted(narrative_community_docs.keys())):
    row = tfidf2[i].toarray().flatten()
    top_idx = row.argsort()[::-1][:12]
    keywords = ", ".join(feature_names2[j] for j in top_idx)
    size = (community_ids_narrative == comm_id).sum()
    print(f"C{comm_id} (n={size:>4}): {keywords}")


TF-IDF keywords per community

C0 (n=  74): bankruptcy court, bankruptcy credit, bankruptcies, bankruptcy, public records, discharged, chapter xxxx, public record, xxxx bankruptcy, reinvestigation, chapter, courts
C1 (n= 403): barclay, kohl, barclays, american express, amex, bank west, elan, west loans, apple, discover card, applecard, boh
C2 (n=  58): capital, 2022 39, 2022 110, xxxx xxxxxx, xxxxxx, xxxxxx xx, capitol, mobile phone, called capital, pii, mr xxxx, capital bank
C3 (n=  90): chase credit, chase, called chase, account chase, xxxx chase, chase xx, contacted chase, chase employee, chase bank, slate, chase savings, chases
C4 (n=  75): citibank, citi, concession, called citibank, zero points, 25 rate, 00 bonus, citi bank, citibank credit, bonus, xxxx citibank, citibank xx
C5 (n= 316): chime, crosscheck, cash app, america, bank america, zelle, schwab, bank manager, bofa, check fraud, personal check, etrade
C6 (n= 339): inquiry date, xxxx inquiry, removal date, unauthorized inqu

In [27]:
# Most central complaint per community
print("Most central complaint per community\n" + "="*65)

for comm_id in sorted(set(community_ids_narrative)):
    nodes_in_comm = [n for n, c in partition_narrative.items() if c == comm_id]
    subG = G_narrative.subgraph(nodes_in_comm)
    central_node = max(subG.degree(), key=lambda x: x[1])[0]
    size = len(nodes_in_comm)
    print(f"\nC{comm_id} ({size} complaints) — node {central_node}")
    print(narrative_texts[central_node][:350])
    print("-"*65)


Most central complaint per community

C0 (74 complaints) — node 2148
Remove CHAPTER XXXX BANKRUPTCY DISCHARGE : XXXX. This is not my bankruptcy, you've got the wrong person here.. The court itself do not furnish any of this information to the credit bureaus. They do not provide any proof to justify their reporting. XXXX sold this information to the credit bureaus which is illegal.. Please get this public information
-----------------------------------------------------------------

C1 (403 complaints) — node 9
In XXXX of XXXX, my card number was stolen and many instances of fraud were captured. Bank of America 's system alerted my phone to the transactions and I responded saying none of them were mine and to cancel them. 

A few transactions on XX/XX/XXXX were not refunded including {$240.00} at XXXX XXXX and {$74.00} at XXXX. I called Bank of America to
-----------------------------------------------------------------

C2 (58 complaints) — node 466
I purchased a single team subscripti

In [28]:
import requests
import json as json_lib
import re

OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL_NAME = "llama3.2:3b"

def get_representative_texts(G, partition, comm_id, texts, n=3):
    nodes_in_comm = [node for node, c in partition.items() if c == comm_id]
    subG = G.subgraph(nodes_in_comm)
    top_nodes = sorted(subG.degree(), key=lambda x: -x[1])[:n]
    return [texts[node] for node, _ in top_nodes]

def characterise_community(keywords, representative_texts):
    excerpts = "\n\n".join(f"- {t[:400]}" for t in representative_texts)
    prompt = f"""You're looking at one cluster from a semantic clustering of consumer financial complaints.

Top distinctive keywords for this cluster:
{keywords}

A few representative complaints from this cluster:
{excerpts}

Describe what this cluster of complaints is about. Respond with ONLY a JSON object, no other text, no markdown formatting:
{{"label": "3-6 word label", "description": "1-2 sentence description of the pattern, written for an investigator"}}"""

    response = requests.post(OLLAMA_URL, json={
        "model": MODEL_NAME,
        "prompt": prompt,
        "stream": False,
        "options": {"temperature": 0.2}
    })
    response.raise_for_status()
    raw = response.json()["response"].strip()

    try:
        return json_lib.loads(raw)
    except json_lib.JSONDecodeError:
        pass

    match = re.search(r"\{.*\}", raw, re.DOTALL)
    if match:
        return json_lib.loads(match.group())

    raise ValueError(f"Could not parse JSON from: {raw[:200]}")

In [29]:
community_descriptions = {}
comm_ids_sorted = sorted(narrative_community_docs.keys())

for idx, comm_id in enumerate(comm_ids_sorted):
    row = tfidf2[idx].toarray().flatten()
    top_idx = row.argsort()[::-1][:10]
    keywords = ", ".join(feature_names2[j] for j in top_idx)

    reps = get_representative_texts(G_narrative, partition_narrative, comm_id, narrative_texts, n=3)

    try:
        result = characterise_community(keywords, reps)
    except (ValueError, json_lib.JSONDecodeError, requests.RequestException) as e:
        result = {"label": "parse error", "description": str(e)}

    community_descriptions[comm_id] = result

    size = (community_ids_narrative == comm_id).sum()
    print(f"C{comm_id} (n={size}): {result.get('label', '?')}")
    print(f"  {result.get('description', '?')}\n")

C0 (n=74): Bankruptcy Credit Errors
  Consumers complaining about inaccurate bankruptcy information in their credit reports, claiming it was obtained without consent from the court.

C1 (n=403): Bank of America issues
  Complaints about unauthorized charges and fraudulent activity on Bank of America credit cards.

C2 (n=58): Capital One billing issues
  Complaints about unauthorized charges and account closures on Capital One accounts, often related to online purchases or unexpected transactions.

C3 (n=90): Chase Customer Service Issues
  Complaints about Chase's handling of customer service calls and responses to security alerts.

C4 (n=75): Citibank Chargeback Issues
  Complaints centered around Citibank's handling of chargebacks, including incorrect balance updates and unauthorized transactions.

C5 (n=316): Bank Account Issues
  Complaints about bank account closures, unauthorized transactions, and difficulties with banking services.

C6 (n=339): Unauthorized Credit Inquiries
  Co

In [31]:
df_narrative["community_label"] = df_narrative["community"].map(
    lambda c: community_descriptions[c].get("label", "unlabelled")
)
df_narrative["community_description"] = df_narrative["community"].map(
    lambda c: community_descriptions[c].get("description", "")
)

In [32]:
print(sorted(set(community_ids_narrative)))
print(len(set(community_ids_narrative)))

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]
17


## Save

Save the narrative corpus with community assignments for use in notebook 07.


In [ ]:
import pickle, os

OUT_DIR = "../data/processed"
os.makedirs(OUT_DIR, exist_ok=True)

df_narrative.to_parquet(os.path.join(OUT_DIR, "narrative_complaints.parquet"), index=False)

with open(os.path.join(OUT_DIR, "narrative_embeddings.pkl"), "wb") as f:
    pickle.dump(narrative_embeddings, f)

with open(os.path.join(OUT_DIR, "narrative_umap.pkl"), "wb") as f:
    pickle.dump(embedding_2d, f)

with open(os.path.join(OUT_DIR, "narrative_partition.pkl"), "wb") as f:
    pickle.dump(partition_narrative, f)

print("Saved:")
print(f"  narrative_complaints.parquet  ({len(df_narrative):,} rows)")
print(f"  narrative_embeddings.pkl      {narrative_embeddings.shape}")
print(f"  narrative_umap.pkl            {embedding_2d.shape}")
print(f"  narrative_partition.pkl       {n_communities_narrative} communities")


## What did we find?

Once template filers were stripped out, Louvain found 17 communities in the remaining ~2,500 narrative complaints. More granular than expected. The local SLM (llama3.2:3b via Ollama) labelled all 17 cleanly, no parse failures.

A lot of these split by company rather than by issue. Bank of America, Chase, Citibank, Wells Fargo, PNC, Capital One, Coinbase all got their own cluster. Makes sense — once the legal-template noise is gone, the dominant signal left is often "what happened to me at this specific institution."

Biggest community is mortgage servicing (441 complaints): servicers failing to communicate, errors in loan modifications, foreclosure notices going out when they shouldn't. Auto loans (199) and bank account issues (316) are the other large ones.

Two clusters are the most fraud-relevant: unauthorized credit inquiries (339, people finding inquiries they never made) and Coinbase account/security issues (27, smaller but worth a closer look).

So: 15 communities from the full template+narrative mix in notebook 05, narrowed to 17 cleaner ones once the legal boilerplate was removed. Product taxonomy never explained any of this. Company name and specific situation did.

Next: the investigator workflow. Given one complaint, find its community, pull the description, check for anomaly signals.